In [3]:
import pandas as pd
df2 = pd.read_csv('../data/dataset-tickets-multi-lang-4-20k.csv')
print(df2.shape)
print(df2.columns.tolist())

(20000, 15)
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


In [4]:
df_en = df2[df2['language'] == 'en']
print(df_en.shape)

print(df_en['type'].value_counts())
print()
print(df_en['queue'].value_counts())

(11923, 15)
type
Incident    4642
Request     3498
Problem     2498
Change      1285
Name: count, dtype: int64

queue
Technical Support                  3412
Product Support                    2232
Customer Service                   1859
IT Support                         1391
Billing and Payments               1302
Returns and Exchanges               582
Service Outages and Maintenance     442
Sales and Pre-Sales                 330
Human Resources                     205
General Inquiry                     168
Name: count, dtype: int64


In [5]:
for cat in df_en['queue'].unique():
    print(f"\n=== {cat} ===")
    for text in df_en[df_en['queue'] == cat]['body'].sample(min(3, len(df_en[df_en['queue']==cat])), random_state=1):
        print("-", text[:150])


=== Customer Service ===
- Dear Customer Support, I am inquiring about the integration options for the Google Nest Wifi Router with our SaaS project management tools. Could you 
- Could you provide information on digital strategies for promoting brand growth? I'm interested in learning about the company's approaches to online ma
- We've noticed a substantial decline in engagement for our digital marketing campaigns, which might be related to compatibility issues with the recent 

=== Technical Support ===
- Hello Customer Support, I am writing to seek details about the digital tactics your firm employs for brand expansion and advancement. Could you kindly
- Respected Customer Support, I am contacting you to address a problem with the Jenkins build that has failed owing to a Git integration error. It is po
- Our team is in need of detailed guidance concerning the implementation and integration strategies for our project management tools. Could you share mo

=== IT Support ===
- Seeking

In [9]:
print(df_en['body'].isna().sum())
df_en = df_en.dropna(subset=['queue'])
df_en['body'] = df_en['body'].fillna('')

1


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

x=df_en['body']
y=df_en['queue']

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
clt=LogisticRegression(max_iter=1000,class_weight='balanced')
clt.fit(X_train_vec, y_train)
print(classification_report(y_test, clt.predict(X_test_vec)))

                                 precision    recall  f1-score   support

           Billing and Payments       0.70      0.69      0.69       260
               Customer Service       0.30      0.25      0.27       372
                General Inquiry       0.12      0.38      0.18        34
                Human Resources       0.26      0.61      0.36        41
                     IT Support       0.35      0.38      0.37       278
                Product Support       0.43      0.29      0.35       447
          Returns and Exchanges       0.22      0.43      0.29       116
            Sales and Pre-Sales       0.13      0.38      0.20        66
Service Outages and Maintenance       0.30      0.55      0.39        88
              Technical Support       0.52      0.32      0.40       683

                       accuracy                           0.37      2385
                      macro avg       0.33      0.43      0.35      2385
                   weighted avg       0.42      0

In [12]:
print (X_test_vec[:5])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 147 stored elements and shape (5, 4676)>
  Coords	Values
  (0, 36)	0.21269563582651416
  (0, 133)	0.11649743551547868
  (0, 299)	0.07326606017895868
  (0, 402)	0.12042862385646765
  (0, 966)	0.07193860110952924
  (0, 1108)	0.0896924104971159
  (0, 1223)	0.15247797054758555
  (0, 1388)	0.10739753188061847
  (0, 1505)	0.16290776299967444
  (0, 1586)	0.13017873968526594
  (0, 1806)	0.19927395601038372
  (0, 1859)	0.09086359169278754
  (0, 2132)	0.1278515297296264
  (0, 2133)	0.1866762268408274
  (0, 2151)	0.1182225760185632
  (0, 2265)	0.20506851389904893
  (0, 2328)	0.13191782710586927
  (0, 2357)	0.23045801388938636
  (0, 2362)	0.17190929528308488
  (0, 2447)	0.20246325356959236
  (0, 2533)	0.16720304453973997
  (0, 2812)	0.22274811147061335
  (0, 2819)	0.3134752208882401
  (0, 2881)	0.23410024646867328
  (0, 3179)	0.1268347688536014
  :	:
  (4, 2107)	0.22067800538691965
  (4, 2159)	0.20779332925955735
  (4, 2220)	0.139044434

In [16]:
df_en['answer'].isna().sum()

np.int64(3)

In [17]:
df_en['body'].isna().sum()
df_en = df_en.dropna(subset=['body'])

In [18]:
kb = df_en.dropna(subset=['answer']).copy()
print(kb.shape)

(11920, 15)


In [13]:
from sentence_transformers import SentenceTransformer
embedder=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
kb_texts = kb['body'].tolist()
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True)

Batches:   0%|          | 0/373 [00:00<?, ?it/s]

In [20]:
print(kb_embeddings.shape)

(11920, 384)


In [21]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="support_tickets")

In [22]:
print("we are making the database in chromadb and inserting the knowledge base into it")
print("we are also grouping similar tickets together based on their queue and storing that information in the metadata")
client.delete_collection(name="support_tickets")
collection = client.create_collection(name="support_tickets")

batch_size = 5000

for start in range(0, len(kb), batch_size):
    end = start + batch_size
    collection.add(
        embeddings=kb_embeddings[start:end].tolist(),
        documents=kb["answer"].tolist()[start:end],
        ids=[str(i) for i in range(start, min(end, len(kb)))],
        metadatas=[{'queue': q} for q in kb['queue']][start:end]
    )
    print(f"Inserted rows {start} to {end}")

we are making the database in chromadb and inserting the knowledge base into it
we are also grouping similar tickets together based on their queue and storing that information in the metadata
Inserted rows 0 to 5000
Inserted rows 5000 to 10000
Inserted rows 10000 to 15000


In [23]:
print(collection.count())

11920


In [24]:
print("we are making a test ticket and seeing if it gives 3 similar tickets from the knowledge base")
new_ticket="My subscription was charged twice this month and i want a refund for the extra charge."
new_ticket_embedding = embedder.encode([new_ticket])
results = collection.query(
    query_embeddings=new_ticket_embedding.tolist(),
    n_results=3
)
print(results)


we are making a test ticket and seeing if it gives 3 similar tickets from the knowledge base
{'ids': [['9526', '11392', '7895']], 'embeddings': None, 'documents': [['We will investigate the duplicate charge issue and process the necessary refund. Please allow us to review your account information (<acc_num>) to resolve the matter promptly.', '<name>, we apologize for the issue with your subscription renewal payment. We understand you have already tried contacting support and reviewing your billing history. We would like to assist you and investigate the matter. Could you please provide your <acc_num> and the date of the duplicate charge? We will work on refunding the extra amount as soon as possible. Please call <tel_num> at your convenience to discuss further.', 'We will investigate the issue with the duplicate charges and process the necessary refund. Please allow us some time to review your account information (<acc_num>) and resolve the matter promptly.']], 'uris': None, 'included'

In [25]:
print("we are getting the api key from the .env file")
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")

print(api_key is not None)

we are getting the api key from the .env file
True


In [26]:
print("we are getting the free models from openrouter.ai")
import requests

response = requests.get("https://openrouter.ai/api/v1/models")
models = response.json()['data']

free_models = [m['id'] for m in models if m['pricing']['prompt'] == '0']
print(free_models)

we are getting the free models from openrouter.ai
['inclusionai/ling-3.0-flash-vl:free', 'nex-agi/nex-n2.5-mini:free', 'nex-agi/nex-n2.5-pro:free', 'inclusionai/ling-3.0-flash-sante:free', 'inclusionai/ling-3.0-flash-fin:free', 'dots-studio/dots-3-note-preview:free', 'liquid/lfm-2.5-2.6b:free', 'nvidia/nemotron-3.5-lightning:free', 'thinkingmachines/inkling-small:free', 'poolside/laguna-s-2.1:free', 'thinkingmachines/inkling:free', 'poolside/laguna-xs-2.1:free', 'cohere/north-mini-code:free', 'nvidia/nemotron-3.5-content-safety:free', 'nvidia/nemotron-3-ultra-550b-a55b:free', 'nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free', 'google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'google/lyria-3-pro-preview', 'google/lyria-3-clip-preview', 'nvidia/nemotron-3-super-120b-a12b:free', 'openrouter/free']


In [27]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
FREE_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working."}
    ]
)

print(response.choices[0].message.content)


Hello! I'm here and working properly. How can I assist you today?


In [28]:
print("the models often have rate limits and dont work but when retried do work we are writing a replay function")
import time

def call_llm_with_retry(model, messages, max_retries=3, wait_seconds=5):
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            if attempt < max_retries:
                print(f"Waiting {wait_seconds} seconds before retrying...")
                time.sleep(wait_seconds)
            else:
                print("Giving up after max retries.")
                raise

the models often have rate limits and dont work but when retried do work we are writing a replay function


In [29]:
reply = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": "Say hello and confirm you're working."}])
print(reply)

Hello! I'm working and ready to help.


In [30]:
def resolution_agent(new_ticket_text, n_results=3):
    query_embedding = embedder.encode([new_ticket_text])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_results
    )
    retrieved_resolutions = results['documents'][0]

    context = "\n\n".join([f"Past resolution {i+1}: {r}" for i, r in enumerate(retrieved_resolutions)])

    prompt = f"""You are a customer support agent. A new ticket has come in:

"{new_ticket_text}"

Here are similar past resolutions for reference:

{context}

Write a helpful, professional reply to the new ticket, using the past resolutions as guidance. Do not just copy them — tailor the reply to this specific ticket."""

    reply = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return reply

In [31]:
draft = resolution_agent("My subscription was charged twice this month and I want a refund")
print(draft)

Hello,

Thank you for bringing this to our attention. I’m sorry to hear that your subscription was charged twice this month, and I understand how frustrating that can be. Rest assured, we’ll look into the duplicate charge right away and process a refund for the extra amount as soon as we confirm the details.

To get started, could you please provide the following information so we can locate your account and review the transactions?

- Your account number (<acc_num>)
- The exact dates of the two charges you saw this month
- Any reference numbers or receipts you may have received

Once we have this information, we’ll verify the duplicate charge and initiate the refund to your original payment method. If you prefer to discuss this over the phone, you can reach our billing support team at <tel_num> during our regular business hours.

We appreciate your patience and will keep you updated on the progress. Please let us know if there’s anything else we can assist you with.

Best regards,  
[

In [32]:
def escalation_agent(new_ticket_text, priority, draft_reply):
    prompt = f"""You are reviewing a customer support ticket and a draft reply before it gets sent.

Ticket: "{new_ticket_text}"
Priority: {priority}
Draft reply: "{draft_reply}"

Decide whether this reply should be sent automatically, or escalated to a human agent for review. Consider: is the priority high/critical, does the draft reply seem confident and complete, or does it seem uncertain or risky to send without human review?

Respond in this exact format:
Decision: [AUTO-SEND or ESCALATE]
Reason: [one sentence explaining why]"""

    decision = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return decision

In [33]:
decision = escalation_agent(
    "My subscription was charged twice this month and I want a refund",
    "high",
    draft
)
print(decision)

Decision: AUTO-SEND
Reason: The draft reply is courteous, provides clear next steps, asks only for necessary information, and does not contain any uncertain or risky statements that would require human review despite the high priority.


In [34]:
from typing import TypedDict, List

class TicketState(TypedDict):
    ticket_text: str
    predicted_queue: str
    retrieved_resolutions: List[str]
    draft_reply: str
    escalation_decision: str

In [35]:
def classifier_node(state: TicketState):
    text = state["ticket_text"]

    ticket_vec = vectorizer.transform([text])

    predicted_queue = clt.predict(ticket_vec)[0]

    return {
        "predicted_queue": predicted_queue
    }

In [36]:
def retriever(state: TicketState) -> TicketState:
    text = state['ticket_text']
    predicted_queue = state['predicted_queue']

    query_embedding = embedder.encode([text])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3,
        where={"queue": predicted_queue}
    )

    retrieved = results['documents'][0]
    return {"retrieved_resolutions": retrieved}

In [37]:
test_state = {"ticket_text": "My subscription was charged twice this month and I want a refund"}
test_state.update(classifier_node(test_state))
test_state.update(retriever(test_state))
print(test_state)

{'ticket_text': 'My subscription was charged twice this month and I want a refund', 'predicted_queue': 'Billing and Payments', 'retrieved_resolutions': ['We will investigate the duplicate charge issue and process the necessary refund. Please allow us to review your account information (<acc_num>) to resolve the matter promptly.', '<name>, we apologize for the issue with your subscription renewal payment. We understand you have already tried contacting support and reviewing your billing history. We would like to assist you and investigate the matter. Could you please provide your <acc_num> and the date of the duplicate charge? We will work on refunding the extra amount as soon as possible. Please call <tel_num> at your convenience to discuss further.', 'Dear [Name], we are looking into the issue with your monthly subscription charges. To assist further, could you please provide us with your account details, specifically [acc_num]? We would also like to verify other relevant information.

In [38]:
from typing import TypedDict, List

class TicketState(TypedDict):
    ticket_text: str
    priority: str
    predicted_queue: str
    retrieved_resolutions: List[str]
    draft_reply: str
    escalation_decision: str

In [39]:
def resolution_node(state: TicketState) -> TicketState:
    ticket_text = state['ticket_text']
    retrieved = state['retrieved_resolutions']

    context = "\n\n".join([f"Past resolution {i+1}: {r}" for i, r in enumerate(retrieved)])

    prompt = f"""You are a customer support agent. A new ticket has come in:

"{ticket_text}"

Here are similar past resolutions for reference:

{context}

Write a helpful, professional reply to the new ticket, using the past resolutions as guidance. Do not just copy them — tailor the reply to this specific ticket."""

    reply = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return {"draft_reply": reply}

In [40]:
def escalation_node(state: TicketState) -> TicketState:
    ticket_text = state['ticket_text']
    priority = state['priority']
    draft_reply = state['draft_reply']

    prompt = f"""You are reviewing a customer support ticket and a draft reply before it gets sent.

Ticket: "{ticket_text}"
Priority: {priority}
Draft reply: "{draft_reply}"

Decide whether this reply should be sent automatically, or escalated to a human agent for review.

Respond in this exact format:
Decision: [AUTO-SEND or ESCALATE]
Reason: [one sentence explaining why]"""

    decision = call_llm_with_retry(FREE_MODEL, [{"role": "user", "content": prompt}])
    return {"escalation_decision": decision}

In [44]:
from langgraph.graph import StateGraph, END

graph = StateGraph(TicketState)

graph.add_node("classifier_node", classifier_node)
graph.add_node("retriever", retriever)
graph.add_node("resolution", resolution_node)
graph.add_node("escalation", escalation_node)

graph.set_entry_point("classifier_node")
graph.add_edge("classifier_node", "retriever")
graph.add_edge("retriever", "resolution")
graph.add_edge("resolution", "escalation")
graph.add_edge("escalation", END)

app = graph.compile()

In [45]:
result = app.invoke({
    "ticket_text": "My subscription was charged twice this month and I want a refund",
    "priority": "high"
})

print(result)

{'ticket_text': 'My subscription was charged twice this month and I want a refund', 'priority': 'high', 'predicted_queue': 'Billing and Payments', 'retrieved_resolutions': ['We will investigate the duplicate charge issue and process the necessary refund. Please allow us to review your account information (<acc_num>) to resolve the matter promptly.', '<name>, we apologize for the issue with your subscription renewal payment. We understand you have already tried contacting support and reviewing your billing history. We would like to assist you and investigate the matter. Could you please provide your <acc_num> and the date of the duplicate charge? We will work on refunding the extra amount as soon as possible. Please call <tel_num> at your convenience to discuss further.', 'Dear [Name], we are looking into the issue with your monthly subscription charges. To assist further, could you please provide us with your account details, specifically [acc_num]? We would also like to verify other r

In [46]:
test_sample = df_en.loc[X_test.index].sample(15, random_state=42)
test_sample[['body', 'queue', 'priority']]

,body,queue,priority
17521,"Hello Customer Support, I am encountering diff...",Technical Support,medium
16014,Requesting enhancements in integration feature...,Product Support,medium
10068,"Hello Customer Support, I am inquiring about t...",Billing and Payments,low
626,An unexpected data leak has happened. It is su...,IT Support,medium
11328,"IntelliJ IDEA experienced a malfunction, disru...",IT Support,high
10394,"Dear Customer Support, I am reaching out to in...",Human Resources,low
4487,"Dear Customer Support, I am reaching out to in...",Customer Service,high
1732,"Respected Customer Support, I am contacting yo...",Product Support,medium
11391,"Dear Customer Support, I am writing to get mor...",Billing and Payments,high
19311,I am contacting you to address a problem I enc...,IT Support,low


In [47]:
eval_results = []

for idx, row in test_sample.iterrows():
    result = app.invoke({
        "ticket_text": row['body'],
        "priority": row['priority']
    })
    result['true_queue'] = row['queue']
    eval_results.append(result)
    print(f"Processed ticket {idx}")

Processed ticket 17521
Processed ticket 16014
Processed ticket 10068
Processed ticket 626
Processed ticket 11328
Processed ticket 10394
Processed ticket 4487
Processed ticket 1732
Processed ticket 11391
Processed ticket 19311
Processed ticket 2277
Processed ticket 18272
Processed ticket 7002
Processed ticket 16769
Processed ticket 2449
